# Polynomial Regression

### Guessing fish age from its length


<img src="./img/4_bluegill.jpeg" width="500px">

Imagine a dataset containing

<table>
    <tr>
        <th>field name</th>
        <th>data</th>
    </tr>
    <tr>
        <td>age</td>
        <td>age of the bluegill (in years)</td>
    </tr>
    <tr>
        <td>length</td>
        <td>length of the fish (in mm)</td>
    </tr>
</table>
<br>
<span style="font-size: 70%">Bluegills are North American freshwater fish, up to 30cm long and 2kg in weight, see image above.</span>
<br><br>

If you catch a bluegill, you may want to know its age, given its length.
<br><br>

In [ ]:
# install dependencies (statsmodels, required by seaborn residual plots)
sm_existed = False
try:
    import statsmodels as sm
    del sm
    sm_existed = True
except ImportError:
    ! uv pip install statsmodels


In [ ]:
# imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import linear_model
from sklearn.metrics import r2_score
import seaborn as sns

In [ ]:
# get the data
df = pd.read_csv("./data/bluegills.csv")

In [ ]:
# analyze visually
sns.set_theme(style="darkgrid")
g = sns.jointplot(x=df.length, y=df.age, kind='reg', color='b')

In [ ]:
g = sns.residplot(x=df.age, y=df.length, lowess=True)

While the jointplot might give us the impression that the data follows linear logic, the residual plot shows unlinearity in error distribution.

$\rightarrow$ Data is non-linear.

What happens if we try a linear model?

In [ ]:
# prepare data
X = df.length.to_numpy().reshape(-1, 1)
y = df.age.to_numpy().reshape(-1, 1)

# create estimator and fit model
lr = linear_model.LinearRegression()
lr.fit(X, y)

print(f"LR intercept: {lr.intercept_[0]:.3f}, coefficient: {lr.coef_[0][0]:.3f}")
print(f"Regression score: {lr.score(X, y):.3f}")


The $R^2$ score (0.711) suggests significance determination. However, this __does not indicate linearity__.

In [ ]:
# visualize
X_pred = np.array((60, 180)).reshape(-1, 1)
y_pred = lr.predict(X_pred)

plt.scatter(X, y)
plt.plot(X_pred, y_pred, c='red')
plt.title('Age by length (linear approximation)')
plt.xlabel('length [mm]')
plt.ylabel('age [y]')
plt.show()


The linear model does not fit optimally.

### The polynomial model

Let's assume that:

> &nbsp;  
> $y_i = \theta_0 + \theta_2 x_i^2 + \epsilon$
> <br><br>

<span style="font-size: 70%"><b>Note:</b> We ignore <em>&theta;<sub>1</sub>x</em> in this case to demonstrate the quadratic nature of growth in bluegills.</span>
<br><br>

In [ ]:
# add a squared length column
df['squared_length'] = df.length ** 2
df

In [ ]:
# prepare data
# we use pandas for data handling and manipulation and convert to numpy when appropriate

X2 = df[['squared_length']].to_numpy()

# create model
pr = linear_model.LinearRegression()
pr.fit(X2, y)

# print parameters and R2
print(f"PR intercept:     {pr.intercept_[0]:.3f}, coefficient: x^2: {pr.coef_[0][0]:.8f}")
print(f"Regression score: {pr.score(X2, y):.3f}")

In [ ]:
# visualize regression
X2_pred = np.linspace(60, 180, 31).reshape(-1, 1)
y2_pred = pr.predict(X2_pred ** 2)                      # correct for the squared length

# plot
plt.scatter(X, y)
plt.plot(X2_pred, y2_pred, c='red')
plt.title('Age by length (polynomial approximation - $x^2$)')
plt.xlabel('length [mm]')
plt.ylabel('age [y]')
plt.show()

The squared model fits better.

In [ ]:
g = sns.residplot(x=df.age, y=df.length, order=2, lowess=True)

The residues are distributed more evenly, the lowess indicator is linear around 0.
<br><br>

### Conclusion

1. LinearRegression, in its simplest form, fits a linear model to the dataset by adjusting a set of parameters in order to make the sum of the squared residuals of the model as small as possible.

2. Linear models can be trained on nonlinear functions of the data. A simple linear regression can be extended by constructing polynomial features from the coefficients.

   A model:

   $y \left( \theta, x \right) = \theta_0 + \theta_1 x_1 + \theta_2 x_2 $

   may be fit to parabolic data using:

   $y \left( \theta, x \right) = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \theta_3 x_1 x_2 + \theta_4 x_1^2 + \theta_5 x_2^2$

3. If the data is not polynomic or lacks certain degree, the training process will eliminate the corresponding $\theta_i$'s.

<br><br>

<table>
<tr>
<td style="border-style: none"><img src="./img/0_reference.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><u>Further reading:</u>
<ul>
<li><a href="https://scikit-learn.org/stable/tutorial/statistical_inference/supervised_learning.html#linear-regression">Linear regression</a></li>
<li><a href="https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression">Linear regression (description)</a></li>
<li><a href="https://scikit-learn.org/stable/modules/linear_model.html#polynomial-regression-extending-linear-models-with-basis-functions">Polynomial regression</a></li>
</ul>
</td>
</tr>
</table>

### Using `Polynomial features`

In the above example, we assumed a squared function to best estimate the bluegill dataset.

Scikit learn offers a function to generalize polynomial regression: `PolynomialFeatures`.

We will apply general polynoms to a sample dataset:

> &nbsp;  
> $y = \theta_0 + \theta_1 x + \theta_2 x^2 + ... + \theta_n x^n + \epsilon = \sum_{i=0}^n \theta_i x^i + \epsilon$
> <br><br>



In [ ]:
# generate sample data
X = np.arange(0, 30)
y = [3, 4, 5, 7, 10, 8, 9, 10, 10, 23, 27, 44, 50, 63, 67, 60, 62, 70, 75, 88, 81, 87, 95, 100, 108, 135, 151, 160, 169, 179]

In [ ]:
# visualize
plt.figure(figsize=(10, 6))
plt.scatter(X, y)
plt.title("Some sample data")
plt.show()

In [ ]:
# 1. Determine the degree of the polynome

# import PolynomialFeatures
from sklearn.preprocessing import PolynomialFeatures

# do not include a bias column
poly = PolynomialFeatures(degree = 2, include_bias=False)

In [ ]:
# 2. Create new features

# In the example above we manually created a squared_length column.
# PolynomialFeatures does this automatically for us.

poly_features = poly.fit(X.reshape(-1, 1))
poly_features = poly_features.transform(X.reshape(-1, 1))
poly_features

In [ ]:
# poly_features is an array that contains the $x$-values and their respective orders.
# As fit and transform are usually executed in sequence, Scikit learn offers an abbreviation

poly_features = poly.fit_transform(X.reshape(-1, 1))

In [ ]:
# 3. Create polynomial regression model

# if you haven't done so far
from sklearn.linear_model import LinearRegression

poly_reg = LinearRegression()

# train the model
poly_reg.fit(poly_features, y)

y_poly_pred = poly_reg.predict(poly_features)
y_poly_pred

In [ ]:
# visualize
plt.figure(figsize=(10, 6))
plt.scatter(X, y)
plt.plot(X, y_poly_pred, c='red')
plt.title("Some sample data and a polynomial regression")
plt.show()

<table>
<tr>
<td style="border-style: none"><img src="./img/0_students_input.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><h5>Students task:</h5>Experiment with the polynomial degree (try ranges 2 ... 5).<br>When does the model overfit the data?</td>
</tr>
</table>

# A word on model automation

In `1a Introduction to DS` we learned that in order to make our analysis reproducable, we should create automated models.

So far, we have developed our tools and results step by step.  
In a real-world project, you would write a wrapper around the steps and make sure they catch errors and are themselves free of errors.

In [ ]:
# put your required imports at the beginning of you calculation

import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# make sure, your data is ready

# sample data generated using this formula:
#   np.random.seed(1)
#   x_1 = np.absolute(np.random.randn(100, 1) * 10)
#   x_2 = np.absolute(np.random.randn(100, 1) * 30)
#   y = 2*x_1**2 + 3*x_1 + 2 + np.random.randn(100, 1)*20
#   np.savetxt('./data/sample_data.csv', fmt='%4f,%4f,%4f')

data = np.loadtxt('./data/sample_data.csv', delimiter=',')
X = data[:,0].reshape(-1, 1)                            # we use only the first column
y = data[:, -1].reshape(-1, 1)

# visualize data
plt.scatter(X, y)
plt.title('Always give a title: Sample Data')
plt.show()

In [ ]:

# prepare polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(X)

# split your data into training-, validation- and test data
# use 80% training, 20% test, random seed 1 (defining a random seed makes results reproducable)
X_train, X_test, y_train, y_test = train_test_split(poly_features, y, test_size=20, random_state=1)

# create and train the model
poly_reg = LinearRegression()
poly_reg.fit(X_train, y_train)

# test again previously unseen data
y_pred = poly_reg.predict(X_test)

# measure quality: Root Mean Squared Error
poly_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Root MSE: {poly_rmse:.3f}")

# describe the model
print(f"Model parameters:\n  Intercept: {poly_reg.intercept_}\n  Coefficients: {poly_reg.coef_}")

# visualize model
X_model = np.linspace(0, 23, 47).reshape(-1, 1)
X_model = poly.fit_transform(X_model)
y_model = poly_reg.predict(X_model)

plt.scatter(X, y)
plt.plot(X_model[:,0], y_model, c='red')
plt.title('Sample Data and polynomial regression model')
plt.savefig('./img/4_sample_data_poly.png')
plt.show()


Congratulations: __28 lines of code__ (including 10 lines for visualization) to create a polynomial model !!!

# Determine model quality

<img src="./img/4_good_enough.png" width="500px">
<br><br>

##### Is the model above good?

We have a Root Mean Squared Error of 20.751. Shouldn't that suffice?

Is there a RMSE of 0? Or 10?
<br><br>

According to our previous theory and with trust in the correctness of the libraries used: __NO__.

Our RMSE is the mininum (the model minimized the loss which, by definition is the MSE).

##### So, are we happy and can go home?

__Not quite yet.__

Our model can be compared to a simple Linear Regression model.

Let's do this.


In [ ]:
# imports
# ... we reuse the imports above

# data
# ... we reuse the data above

# train- / test split
X_train_lr, X_test_lr, y_train_lr, y_test_lr = train_test_split(X, y, test_size=20, random_state=1)

# create and train the model
lr = LinearRegression()
lr.fit(X_train_lr, y_train_lr)

# test model
y_pred_lr = lr.predict(X_test_lr)

# RMSE
lr_rmse = np.sqrt(mean_squared_error(y_test_lr, y_pred_lr))

print(f"Root MSE: {lr_rmse:.3f}")

# describe the model
print(f"Model parameters:\n  Intercept: {lr.intercept_}\n  Coefficients: {lr.coef_}")

# visualize model
X_model = np.linspace(0, 23, 47).reshape(-1, 1)
y_model = lr.predict(X_model)

plt.scatter(X, y)
plt.plot(X_model, y_model, c='red')
plt.title('Sample Data and polynomial regression model')
plt.savefig('./img/4_sample_data_lr.png')
plt.show()


Our polynomial model performs better than a linear model.

<table>
    <tr>
        <td>&nbsp;</td>
        <td>Our model<br>(Polynomial Regression)</td>
        <td>Linear Regression</td>
    </tr>
    <tr>
        <td>Degree</td>
        <td>2</td>
        <td>1</td>
    </tr>
    <tr>
        <td>RMSE</td>
        <td>20.751</td>
        <td>63.541</td>
    </tr>
    <tr>
        <td>Fit</td>
        <td><img src="./img/4_sample_data_poly.png" width="300px"></td>
        <td><img src="./img/4_sample_data_lr.png" width="300px"></td>
    </tr>
</table>

Now we can take a

<span style="font-size: 128px">&#9749;</span> Coffee break!

In [ ]:
# cleanup statsmodels and patsy
if not sm_existed:
    ! pip uninstall -y statsmodels patsy
